# Explorando Instacart: descubre patrones de compra

## Momento 1 · Comprensión del problema

**Problema.** Como analista de datos de Instacart, debo explorar cinco conjuntos de datos (pedidos, productos, pasillos, departamentos y artículos por pedido) para identificar qué información contienen y cómo consultarla.

**Pregunta orientadora.** ¿Cómo usar Python y Pandas para explorar los datos de Instacart, identificar las características de pedidos y productos y obtener información sobre los hábitos de compra?

**Preguntas de análisis (borrador; ajústalas a tu criterio):**
1. ¿A qué hora del día se hacen más pedidos?
2. ¿Qué proporción de los artículos comprados son recompras?
3. ¿Cada cuántos días vuelven a comprar los clientes, en promedio?

**¿Por qué explorar antes de analizar?** Conocer las dimensiones, los tipos de datos y los valores ausentes evita sacar conclusiones erróneas y permite elegir bien las operaciones.

## Momento 2 · Importación y exploración inicial

In [1]:
import pandas as pd

orders = pd.read_csv("../data/instacart_orders.csv", sep=";")
products = pd.read_csv("../data/products.csv", sep=";")
aisles = pd.read_csv("../data/aisles.csv", sep=";")
departments = pd.read_csv("../data/departments.csv", sep=";")
order_products = pd.read_csv("../data/order_products.csv", sep=";")

datos = {
    "orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
}

In [2]:
for nombre, df in datos.items():
    print(f"=== {nombre} ===")
    print("Primeros 5:")
    display(df.head())
    print("Últimos 5:")
    display(df.tail())

=== orders ===
Primeros 5:


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
1,1690866,163593,5,5,12,9.0
2,1454967,39980,4,5,19,2.0
3,1768857,82516,56,0,20,10.0
4,3007858,196724,2,4,12,17.0


Últimos 5:


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
478962,3210681,5617,5,1,14,7.0
478963,3270802,112087,2,3,13,6.0
478964,885349,82944,16,2,11,6.0
478965,216274,4391,3,3,8,8.0
478966,2071924,1730,18,1,14,15.0


=== products ===
Primeros 5:


...

,department_id,department
16,17,household
17,18,babies
18,19,snacks
19,20,deli
20,21,missing


=== order_products ===
Primeros 5:


,order_id,product_id,add_to_cart_order,reordered
0,2141543,11440,17.0,0
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1


Últimos 5:


,order_id,product_id,add_to_cart_order,reordered
4545002,577211,15290,12.0,1
4545003,1219554,21914,9.0,0
4545004,692640,47766,4.0,1
4545005,319435,691,8.0,1
4545006,1398151,28733,9.0,0


In [3]:
resumen = pd.DataFrame(
    {nombre: {"filas": df.shape[0], "columnas": df.shape[1]} for nombre, df in datos.items()}
).T
display(resumen)

print("Más registros:", resumen["filas"].idxmax())
print("Menos columnas:", resumen[resumen["columnas"] == resumen["columnas"].min()].index.tolist())

,filas,columnas
orders,478967,6
products,49694,4
aisles,134,2
departments,21,2
order_products,4545007,4


Más registros: order_products
Menos columnas: ['aisles', 'departments']


### Respuestas del Momento 2

- **¿Cuál tiene más registros?** `order_products`, con más de 4,5 millones de filas.
- **¿Cuál tiene menos columnas?** `aisles` y `departments`, con 2 columnas cada uno.
- **Pedidos vs. productos:** (completa con lo que observes en `head()`: qué describe cada fila de `orders` y de `products`).
- **¿Por qué importan las dimensiones?** (completa con tus palabras: memoria, tiempo de proceso, saber cuántos registros hay antes de filtrar).

## Momento 3 · Estructura y calidad inicial de los datos

In [4]:
for nombre, df in datos.items():
    print(f"=== {nombre} ===")
    print("Columnas:", df.columns.tolist())
    print(df.dtypes)
    print()

=== orders ===
Columnas: ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
order_id                    int64
user_id                     int64
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object

=== products ===
Columnas: ['product_id', 'product_name', 'aisle_id', 'department_id']
product_id       int64
product_name       str
aisle_id         int64
department_id    int64
dtype: object

=== aisles ===
Columnas: ['aisle_id', 'aisle']
aisle_id    int64
aisle         str
dtype: object

=== departments ===
Columnas: ['department_id', 'department']
department_id    int64
department         str
dtype: object

=== order_products ===
Columnas: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']
order_id               int64
product_id             int64
add_to_cart_order    float64
reordered              int64
dtype: object



In [5]:
for nombre, df in datos.items():
    print(f"=== {nombre} ===")
    df.info()
    print()

=== orders ===
<class 'pandas.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_number            478967 non-null  int64  
 3   order_dow               478967 non-null  int64  
 4   order_hour_of_day       478967 non-null  int64  
 5   days_since_prior_order  450148 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 21.9 MB

=== products ===
<class 'pandas.DataFrame'>
RangeIndex: 49694 entries, 0 to 49693
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_id     49694 non-null  int64
 1   product_name   48436 non-null  str  
 2   aisle_id       49694 non-null  int64
 3   department_id  49694 non-null  int64
dtypes: int64(3), str(1)
memory usag

In [6]:
for nombre, df in datos.items():
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0]
    print(nombre, "->", nulos.to_dict() if len(nulos) else "sin valores ausentes")

orders -> {'days_since_prior_order': 28819}
products -> {'product_name': 1258}
aisles -> sin valores ausentes
departments -> sin valores ausentes
order_products -> {'add_to_cart_order': 836}


In [7]:
diagnostico = pd.DataFrame(
    {
        nombre: {
            "filas": df.shape[0],
            "columnas": df.shape[1],
            "valores_ausentes": int(df.isna().sum().sum()),
            "columnas_con_ausentes": ", ".join(df.columns[df.isna().any()]) or "ninguna",
        }
        for nombre, df in datos.items()
    }
).T
display(diagnostico)

,filas,columnas,valores_ausentes,columnas_con_ausentes
orders,478967,6,28819,days_since_prior_order
products,49694,4,1258,product_name
aisles,134,2,0,ninguna
departments,21,2,0,ninguna
order_products,4545007,4,836,add_to_cart_order


### Campos que relacionan los conjuntos de datos

| Campo | Relaciona |
| --- | --- |
| `order_id` | `orders` con `order_products` |
| `product_id` | `products` con `order_products` |
| `aisle_id` | `products` con `aisles` |
| `department_id` | `products` con `departments` |
| `user_id` | identifica al cliente dentro de `orders` |

### Respuestas del Momento 3

- **int64, float64 y object:** `int64` guarda números enteros, `float64` números decimales (y es el tipo que usa Pandas cuando una columna entera tiene valores ausentes) y `object` guarda texto.
- **¿Qué da `info()` que no da `shape`?** Los nombres de columnas, los tipos de dato, el conteo de valores no nulos por columna y el uso de memoria.
- **¿Dónde hay valores ausentes?** (En tres columnas: `days_since_prior_order` en `orders` (28.819 ausentes), `product_name` en `products` (1.258) y `add_to_cart_order` en `order_products` (836). `aisles` y `departments` no tienen ausentes.).
- **¿Qué dificultades pueden causar?** (Los promedios y conteos ignoran las filas con ausentes, así que el promedio de `days_since_prior_order` se calcula solo con los pedidos que sí tienen valor. Además, los ausentes impiden convertir una columna a enteros y los productos sin nombre no se pueden identificar al cruzar tablas, lo que puede sesgar las conclusiones.).

No se corrigen ni se eliminan registros en esta fase: solo se reconocen las limitaciones de los datos.

## Momento 4 · Indexación y filtrado lógico de la información

| N.º | Requerimiento | DataFrame |
| --- | --- | --- |
| 1 | Seleccionar solo `product_id` y `product_name` | `products` |
| 2 | Productos con `department_id` igual a 1 | `products` |
| 3 | Pedidos realizados a las 10 horas | `orders` |
| 4 | Pedidos con `order_dow` igual a 0 | `orders` |
| 5 | Pedidos con `order_number` superior a 10 | `orders` |
| 6 | Pedidos con `days_since_prior_order` superior a 15 | `orders` |
| 7 | Registros con `reordered` igual a 1 | `order_products` |
| 8 | Productos agregados primero al carrito (`add_to_cart_order` igual a 1) | `order_products` |

Después se aplican dos filtros con condiciones simultáneas. En cada ejercicio, la línea impresa antes de la tabla resume el resultado: cuántos registros cumplen la condición y qué porcentaje del total representan.

In [8]:
def mostrar(resultado, total, descripcion):
    n = len(resultado)
    print(f"{descripcion}: {n:,} de {total:,} registros ({n / total:.1%})".replace(",", "."))
    display(resultado.head())

In [9]:
# 1. Seleccionar columnas (products)
seleccion = products[["product_id", "product_name"]]
print("Columnas seleccionadas:", seleccion.shape[1], "| Filas:", f"{seleccion.shape[0]:,}".replace(",", "."))
display(seleccion.head())

Columnas seleccionadas: 2 | Filas: 49.694


,product_id,product_name
0,1,Chocolate Sandwich Cookies
1,2,All-Seasons Salt
2,3,Robust Golden Unsweetened Oolong Tea
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...
4,5,Green Chile Anytime Sauce


In [10]:
# 2. Productos del departamento 1 (products)
r = products.loc[products["department_id"] == 1]
mostrar(r, len(products), "Productos del departamento 1")

Productos del departamento 1: 4.008 de 49.694 registros (8.1%)


,product_id,product_name,aisle_id,department_id
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
29,30,Three Cheese Ziti- Marinara with Meatballs,38,1


In [11]:
# 3. Pedidos realizados a las 10 horas (orders)
r = orders.loc[orders["order_hour_of_day"] == 10]
mostrar(r, len(orders), "Pedidos realizados a las 10 horas")

Pedidos realizados a las 10 horas: 40.578 de 478.967 registros (8.5%)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
28,133707,182261,1,3,10,NaN
36,3378084,127629,21,1,10,20.0
49,2161799,87345,7,0,10,24.0
51,1739741,118405,48,2,10,3.0


In [12]:
# 4. Pedidos con order_dow igual a 0 (orders)
r = orders.loc[orders["order_dow"] == 0]
mostrar(r, len(orders), "Pedidos con order_dow igual a 0")

Pedidos con order_dow igual a 0: 84.090 de 478.967 registros (17.6%)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
3,1768857,82516,56,0,20,10.0
5,1843871,20084,5,0,11,15.0
7,2521283,97711,2,0,18,22.0
23,3406282,19356,21,0,0,14.0
25,1895419,169793,2,0,11,8.0


In [13]:
# 5. Pedidos con order_number superior a 10 (orders)
r = orders.loc[orders["order_number"] > 10]
mostrar(r, len(orders), "Pedidos con order_number superior a 10")

Pedidos con order_number superior a 10: 244.072 de 478.967 registros (51.0%)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
3,1768857,82516,56,0,20,10.0
6,2511640,148233,40,3,8,2.0
8,2266494,34954,40,5,15,4.0
10,1162566,43738,33,4,11,2.0


In [14]:
# 6. Pedidos con days_since_prior_order superior a 15 (orders)
# Las filas con valor ausente (NaN) no cumplen la comparación y quedan fuera.
r = orders.loc[orders["days_since_prior_order"] > 15]
mostrar(r, len(orders), "Pedidos con days_since_prior_order superior a 15")

Pedidos con days_since_prior_order superior a 15: 110.706 de 478.967 registros (23.1%)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
4,3007858,196724,2,4,12,17.0
7,2521283,97711,2,0,18,22.0
14,2843219,74699,4,6,11,30.0
26,1844247,122558,4,6,18,30.0


In [15]:
# 7. Registros con reordered igual a 1 (order_products)
r = order_products.loc[order_products["reordered"] == 1]
mostrar(r, len(order_products), "Artículos con reordered igual a 1")

Artículos con reordered igual a 1: 2.683.838 de 4.545.007 registros (59.1%)


,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1
5,2456893,21616,4.0,1


In [16]:
# 8. Productos agregados primero al carrito (order_products)
r = order_products.loc[order_products["add_to_cart_order"] == 1]
mostrar(r, len(order_products), "Artículos agregados en la primera posición")

Artículos agregados en la primera posición: 450.046 de 4.545.007 registros (9.9%)


,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
14,1961225,37553,1.0,1
16,639939,10017,1.0,1
23,750040,8518,1.0,0


### Filtros con condiciones simultáneas

In [17]:
# 9. Pedidos a las 10 horas Y con order_number superior a 10 (orders)
r = orders.loc[(orders["order_hour_of_day"] == 10) & (orders["order_number"] > 10)]
mostrar(r, len(orders), "Pedidos a las 10 horas y con order_number superior a 10")

Pedidos a las 10 horas y con order_number superior a 10: 21.537 de 478.967 registros (4.5%)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
36,3378084,127629,21,1,10,20.0
51,1739741,118405,48,2,10,3.0
92,60724,110808,78,0,10,2.0
140,3156095,113920,14,2,10,2.0


In [18]:
# 10. Departamento 1 Y aisle_id superior a 50 (products)
r = products.loc[(products["department_id"] == 1) & (products["aisle_id"] > 50)]
mostrar(r, len(products), "Productos del departamento 1 con aisle_id superior a 50")

Productos del departamento 1 con aisle_id superior a 50: 1.618 de 49.694 registros (3.3%)


,product_id,product_name,aisle_id,department_id
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
45,46,Raisin Cinnamon Bagels 5 count,58,1
127,128,Organic Wild Blueberries,116,1


In [19]:
# Comparación: cada condición por separado frente a las dos juntas
c1 = (orders["order_hour_of_day"] == 10).sum()
c2 = (orders["order_number"] > 10).sum()
c12 = ((orders["order_hour_of_day"] == 10) & (orders["order_number"] > 10)).sum()
print(f"Solo a las 10 horas: {c1:,} | Solo order_number > 10: {c2:,} | Ambas: {c12:,}".replace(",", "."))

Solo a las 10 horas: 40.578 | Solo order_number > 10: 244.072 | Ambas: 21.537


### Respuestas del Momento 4

- **Seleccionar columnas vs. filtrar filas:** seleccionar columnas (`products[["product_id", "product_name"]]`) reduce el ancho de la tabla y conserva todas las filas. Filtrar filas (`products.loc[products["department_id"] == 1]`) reduce el largo y conserva todas las columnas.
- **Ventajas de `loc[]`:** permite filtrar filas con una condición y, si se quiere, elegir columnas en la misma instrucción. Además deja explícito que se selecciona por etiquetas o por máscara booleana, y es la forma segura de asignar valores sin ambigüedad.
- **Cantidad de registros con varias condiciones:** con `&` el resultado es siempre menor o igual que el de cada condición por separado, porque solo quedan las filas que cumplen ambas. La última celda de código lo muestra con números.
- **Operadores lógicos:** en Pandas se usan `&` (y), `|` (o) y `~` (no), no `and`/`or`, y cada condición va entre paréntesis por la precedencia de los operadores. Usar el operador equivocado cambia el conjunto de filas seleccionadas o produce un error.
- **Valores ausentes:** en el ejercicio 6, `days_since_prior_order` tiene ausentes y esas filas no cumplen `> 15`, así que quedan fuera del resultado.

## Momento 5 · Obtención e interpretación de indicadores básicos

Antes de calcular, conviene separar tres conceptos que se confunden fácilmente: **registros** (filas de una tabla), **pedidos** (valores distintos de `order_id`) y **usuarios** (valores distintos de `user_id`).

In [20]:
def fmt(x, decimales=0):
    s = f"{x:,.{decimales}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

n_registros = orders.shape[0]
n_pedidos = orders["order_id"].nunique()
n_usuarios = orders["user_id"].nunique()

print("Registros (filas) en orders:", fmt(n_registros))
print("Pedidos distintos (order_id):", fmt(n_pedidos))
print("Usuarios distintos (user_id):", fmt(n_usuarios))

Registros (filas) en orders: 478.967
Pedidos distintos (order_id): 478.952
Usuarios distintos (user_id): 157.437


In [21]:
desc_hora = orders["order_hour_of_day"].describe()
desc_carrito = order_products["add_to_cart_order"].describe()

print("order_hour_of_day")
display(desc_hora)
print("add_to_cart_order")
display(desc_carrito)

order_hour_of_day


count    478967.000000
mean         13.446676
std           4.224986
min           0.000000
25%          10.000000
50%          13.000000
75%          16.000000
max          23.000000
Name: order_hour_of_day, dtype: float64

add_to_cart_order


count    4.544171e+06
mean     8.351669e+00
std      7.080368e+00
min      1.000000e+00
25%      3.000000e+00
50%      6.000000e+00
75%      1.100000e+01
max      6.400000e+01
Name: add_to_cart_order, dtype: float64

In [22]:
n_productos = products.shape[0]
media_orden = orders["order_number"].mean()
media_dias = orders["days_since_prior_order"].mean()
n_dias_validos = orders["days_since_prior_order"].count()
n_dias_ausentes = orders["days_since_prior_order"].isna().sum()
n_recompras = (order_products["reordered"] == 1).sum()
suma_reordered = order_products["reordered"].sum()
pct_recompras = n_recompras / len(order_products)
coincide = n_recompras == suma_reordered

repetidos = n_registros - n_pedidos
nota_repetidos = f" Hay {fmt(repetidos)} filas con order_id repetido." if repetidos > 0 else ""

indicadores = pd.DataFrame([
    {
        "Pregunta": "¿Cuántos registros de pedidos contiene el archivo?",
        "Valor": fmt(n_registros),
        "Interpretación": f"Son filas, no clientes: corresponden a {fmt(n_pedidos)} order_id distintos y {fmt(n_usuarios)} usuarios distintos.{nota_repetidos}",
    },
    {
        "Pregunta": "¿Cuántos registros de productos contiene el catálogo?",
        "Valor": fmt(n_productos),
        "Interpretación": "Cada fila describe un producto del catálogo con su pasillo y su departamento.",
    },
    {
        "Pregunta": "¿Cuál es el promedio de order_number?",
        "Valor": fmt(media_orden, 2),
        "Interpretación": "En promedio, un pedido es el número indicado dentro del historial de su usuario. Es un promedio por pedido, no por usuario, así que pesan más los clientes que compran con frecuencia.",
    },
    {
        "Pregunta": "¿Cuál es el promedio de days_since_prior_order entre los valores disponibles?",
        "Valor": fmt(media_dias, 2) + " días",
        "Interpretación": f"Se calcula solo sobre {fmt(n_dias_validos)} pedidos con valor; los {fmt(n_dias_ausentes)} ausentes no entran en el promedio.",
    },
    {
        "Pregunta": "¿Cuántos registros de artículos tienen reordered igual a 1?",
        "Valor": fmt(n_recompras),
        "Interpretación": f"Son {pct_recompras:.1%} de los {fmt(len(order_products))} artículos de order_products: productos que el cliente ya había comprado antes.".replace(".", ",", 0),
    },
    {
        "Pregunta": "¿Cuál es la suma de la columna reordered?",
        "Valor": fmt(suma_reordered),
        "Interpretación": "Como reordered solo toma los valores 0 y 1, la suma coincide con el conteo anterior." if coincide else "La suma difiere del conteo anterior, lo que indica que reordered toma otros valores además de 0 y 1.",
    },
    {
        "Pregunta": "¿Qué características estadísticas presenta order_hour_of_day?",
        "Valor": f"media {fmt(desc_hora['mean'], 2)} h; mediana {fmt(desc_hora['50%'], 1)} h",
        "Interpretación": f"Los pedidos se reparten entre las {fmt(desc_hora['min'])} y las {fmt(desc_hora['max'])} h; el 50 % central ocurre entre las {fmt(desc_hora['25%'])} y las {fmt(desc_hora['75%'])} h.",
    },
    {
        "Pregunta": "¿Qué características estadísticas presenta add_to_cart_order?",
        "Valor": f"media {fmt(desc_carrito['mean'], 2)}; mediana {fmt(desc_carrito['50%'], 1)}",
        "Interpretación": f"Se calcula sobre {fmt(desc_carrito['count'])} registros con valor ({fmt(len(order_products) - desc_carrito['count'])} ausentes). La posición máxima en el carrito es {fmt(desc_carrito['max'])}.",
    },
])

display(
    indicadores.style.hide(axis="index").set_properties(
        **{"text-align": "left", "white-space": "normal"}
    )
)

Pregunta,Valor,Interpretación
¿Cuántos registros de pedidos contiene el archivo?,478.967,"Son filas, no clientes: corresponden a 478.952 order_id distintos y 157.437 usuarios distintos. Hay 15 filas con order_id repetido."
¿Cuántos registros de productos contiene el catálogo?,49.694,Cada fila describe un producto del catálogo con su pasillo y su departamento.
¿Cuál es el promedio de order_number?,"17,16","En promedio, un pedido es el número indicado dentro del historial de su usuario. Es un promedio por pedido, no por usuario, así que pesan más los clientes que compran con frecuencia."
¿Cuál es el promedio de days_since_prior_order entre los valores disponibles?,"11,10 días",Se calcula solo sobre 450.148 pedidos con valor; los 28.819 ausentes no entran en el promedio.
¿Cuántos registros de artículos tienen reordered igual a 1?,2.683.838,Son 59.1% de los 4.545.007 artículos de order_products: productos que el cliente ya había comprado antes.
¿Cuál es la suma de la columna reordered?,2.683.838,"Como reordered solo toma los valores 0 y 1, la suma coincide con el conteo anterior."
¿Qué características estadísticas presenta order_hour_of_day?,"media 13,45 h; mediana 13,0 h",Los pedidos se reparten entre las 0 y las 23 h; el 50 % central ocurre entre las 10 y las 16 h.
¿Qué características estadísticas presenta add_to_cart_order?,"media 8,35; mediana 6,0",Se calcula sobre 4.544.171 registros con valor (836 ausentes). La posición máxima en el carrito es 64.


### Registros, pedidos y usuarios

Una fila de `orders` es un pedido y un mismo `user_id` aparece en varias filas, porque cada usuario hace varios pedidos. A su vez, `order_products` tiene una fila por cada artículo de cada pedido. Por eso el número de filas de `orders` no es el número de clientes distintos, y el de `order_products` no es el número de pedidos.